# Gang Indictment RAG Notebook

## Goals: 
...

### Create embeddings

In [2]:
import pdfplumber

document_path = "pdfs/Brooklyn_Indictments/8_29_24_9Trey.pdf"
def get_text(document_path):
    with pdfplumber.open(document_path) as pdf:
        # first_page = pdf.pages[0]
        # print(first_page.chars[0]) 
        full_text = ""
        for page in pdf.pages:
            full_text += page.extract_text() + "\n"
        return full_text


In [2]:
text = get_text(document_path)

In [ ]:
from portkey_ai import Portkey 
from dotenv import load_dotenv
import os
import json

load_dotenv()
portkey_api_key = os.getenv('PORTKEY_API_KEY')

portkey = Portkey(
  base_url = "https://ai-gateway.apps.cloud.rt.nyu.edu/v1",
  api_key = portkey_api_key
)


def extract_fields(text):
    response = portkey.chat.completions.create(
    model = "@openai-nyu-it-d-5b382a/gpt-4o-mini",
    messages = [
      {"role": "system", "content": "You are a precise data extractor. Return only valid JSON, no markdown, no backticks."},
      {"role": "user", "content": f"""
Extract the following fields from this press release and return as JSON:
- release_date
- borough
- da_name
- case_summary (1-2 sentences)
- gangs_involved (list of gang names)
- defendants (list of objects with: name, age, aliases, gang_affiliation)
- charges (list of unique charges mentioned)
- incidents (list of objects with: date, location, description)
- locations (list of objects with: address, zip code, precinct, neighborhood, location_type (e.g. "gang territory", "incident location", "housing project"), coordinates (set to null, will geocode later))
- gang_territories (list of objects with: gang_name, description)

If a field is not found, set it to null. For lists, return empty list if nothing found.

Text:
{text}
"""}
    ],
    MAX_TOKENS = 512,
    temperature=0
)
    return json.loads(response.choices[0].message.content)



In [4]:
result = extract_fields(text)
print(json.dumps(result, indent=2))

{
  "release_date": "2024-08-29",
  "borough": "Brooklyn",
  "da_name": "Eric Gonzalez",
  "case_summary": "Eleven alleged members of two opposing street gangs are charged in two indictments with conspiracy to commit murder, possess weapons, and related acts of violence. The defendants are connected to seven shooting incidents, resulting in three people shot, one fatally.",
  "gangs_involved": [
    "Stain Gang",
    "Albany Gang",
    "9Trey",
    "Gorilla Stone",
    "Mac Balla",
    "AMG",
    "59 Brims"
  ],
  "defendants": [
    {
      "name": "Olujimi Lucas",
      "age": 30,
      "aliases": [
        "O Skeeno"
      ],
      "gang_affiliation": "Stain Gang"
    },
    {
      "name": "Jahrell Madison",
      "age": 22,
      "aliases": [
        "Jah Billy"
      ],
      "gang_affiliation": "Stain Gang"
    },
    {
      "name": "Micah Gerson",
      "age": 22,
      "aliases": [
        "Flex"
      ],
      "gang_affiliation": "Stain Gang"
    },
    {
      "name": "Derr